# DataMate & ZAIC 2026 Hybrid Pipeline trên Colab Pro

Notebook này chạy toàn bộ quy trình **Text-to-SQL Hybrid**: kết hợp Ingestion, Catalog, SQL Solver và Rulebook an toàn của **ZAIC_2026** cùng lớp Semantic Views (`v_sales`, `v_orders`), Debt Layer, Dynamic Hints và Few-shot Bank của **DataMate**.

> **Lưu ý quan trọng về Lưu trữ (Google Drive):**
> Notebook được cấu hình **tự động lưu toàn bộ output** (bao gồm file nộp bài `submission.jsonl`, toàn bộ log chi tiết `logs/test_log.jsonl`, `logs/train_log.jsonl`, báo cáo `reports/`, cấu hình lần chạy) trực tiếp ra **Google Drive** của bạn. Dù phiên Colab bị ngắt kết nối (timeout), mọi kết quả và log đều được bảo toàn nguyên vẹn.

## 1. Mount Google Drive, thiết lập thư mục làm việc & cài thư viện

In [ ]:
import os, sys, subprocess, shutil, getpass
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

# CẤU HÌNH GITHUB REPO CỦA BẠN:
# Thay đổi link repo và nhánh nếu bạn push lên repo GitHub riêng
REPO_URL = "https://github.com/TRUNGNT0/datamate_zaloAI.git"   # <-- Thay URL repo của bạn tại đây
BRANCH = "main"
TARGET_DIR = Path("/content/datamate_zaic_2026")

if IN_COLAB:
    # 1. Mount Google Drive để lưu bài nộp, logs và reports lâu dài
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    
    # 2. Tự động tải / Clone mã nguồn từ GitHub vào Colab
    print("▶ Đang lấy mã nguồn dự án từ GitHub...")
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")  # Lấy token từ Colab Secrets (nếu repo private)
    except Exception:
        pass

    clone_url = REPO_URL
    if token:
        clone_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")

    if (TARGET_DIR / ".git").is_dir():
        print(f"Thư mục {TARGET_DIR} đã tồn tại. Đang cập nhật code mới nhất (git pull)...")
        subprocess.run(["git", "pull", "origin", BRANCH], cwd=TARGET_DIR, check=False)
    else:
        print(f"Đang clone {REPO_URL} (branch: {BRANCH})...")
        res = subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", clone_url, str(TARGET_DIR)], capture_output=True, text=True)
        if res.returncode != 0:
            if "Authentication failed" in res.stderr or "could not read Username" in res.stderr:
                token = getpass.getpass("Repo là private! Hãy nhập GitHub Personal Access Token (PAT): ")
                clone_url = REPO_URL.replace("https://", f"https://x-access-token:{token}@")
                subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", clone_url, str(TARGET_DIR)], check=True)
            else:
                raise RuntimeError(f"Lỗi clone Git: {res.stderr}")

    # Kéo LFS nếu repo có file lớn hoặc weights LoRA
    subprocess.run(["git", "lfs", "pull"], cwd=TARGET_DIR, capture_output=True)
    ROOT = TARGET_DIR
    os.chdir(ROOT)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    os.chdir(ROOT)

sys.path.insert(0, str(ROOT))
print(f"Thư mục làm việc: {ROOT}")

# 3. Cài đặt các gói thư viện cần thiết
if IN_COLAB:
    print("Đang cài đặt thư viện từ requirements.txt và requirements-gpu.txt...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-gpu.txt")], check=True)
    print("Cài đặt hoàn tất.")

# 4. Kiểm tra GPU
import torch
if not torch.cuda.is_available():
    print("CẢNH BÁO: Chưa bật GPU! Hãy vào Runtime -> Change runtime type -> Chọn GPU (A100/L4/T4).")
else:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU khả dụng: {gpu_name} | VRAM: {vram_gb:.1f} GB")


## 2. Cấu hình Pipeline & Thư mục Lưu trữ Google Drive

- `DATASET_SPLIT`: `"public_test"` (sinh bài nộp) hoặc `"train"` (chấm điểm PA).
- `DRIVE_SAVE_DIR`: Thư mục cố định trên Google Drive lưu toàn bộ kết quả, log và submission.

In [ ]:
from datamate.local_pipeline import LocalPipelineConfig, LocalSQLPipeline

DATASET_SPLIT = "public_test"          # "public_test" | "train"
MODEL_NAME = "Qwen/Qwen3.5-9B"           # Repo model hoặc đường dẫn checkpoint

# CẤU HÌNH LƯU TRỮ GOOGLE DRIVE:
# Tất cả output, logs, submission.jsonl, reports đều được tự động đồng bộ vào thư mục này trên Drive
if IN_COLAB:
    DRIVE_SAVE_DIR = Path("/content/drive/MyDrive/datamate_zaic_outputs")
else:
    DRIVE_SAVE_DIR = ROOT / "outputs" / "local_runs"

DRIVE_SAVE_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_OUT_DIR = Path("/content/run_tmp" if IN_COLAB else ROOT / "outputs" / "tmp")
LOCAL_OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"▶ Thư mục lưu lâu dài trên Google Drive: {DRIVE_SAVE_DIR}")
print(f"▶ Thư mục chạy tạm thời (Fast NVMe SSD): {LOCAL_OUT_DIR}")

def sync_outputs_to_drive():
    """Đồng bộ tất cả submission, logs và reports từ thư mục chạy tạm thời sang Google Drive."""
    if not IN_COLAB:
        return
    print(f"\n[Sync] Đang lưu toàn bộ logs và output sang Google Drive ({DRIVE_SAVE_DIR})...", flush=True)
    for src_dir in [LOCAL_OUT_DIR, ROOT / "outputs"]:
        if src_dir.exists():
            for item in src_dir.rglob("*"):
                if item.is_file() and not item.name.endswith(".wal") and not item.suffix == ".tmp":
                    rel = item.relative_to(src_dir)
                    dst = DRIVE_SAVE_DIR / rel
                    dst.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy2(item, dst)
    print(f"[Sync] Đã lưu thành công ra Google Drive: {DRIVE_SAVE_DIR}\n", flush=True)

# Khởi tạo cấu hình pipeline
config = LocalPipelineConfig(
    dataset_dir=str(ROOT / "dataset"),
    out_dir=str(LOCAL_OUT_DIR),
    model=MODEL_NAME,
    hf_quant="auto",                  # Tự động nén 4-bit nếu GPU < 24GB để tránh OOM
    max_model_len=16384,
    sql_tokens=2048,
    n_samples=3,                      # Số mẫu SQL độc lập cho mỗi câu hỏi
    agree_min=2,                      # Số phiếu đồng thuận tối thiểu (dưới ngưỡng -> abstain)
    max_retry=2                       # Số lần tự sửa lỗi qua DuckDB feedback
)

SPLITS = (DATASET_SPLIT,)
pipeline = LocalSQLPipeline(config)
print("Khởi tạo LocalSQLPipeline thành công!")


## 3. Chuẩn bị dữ liệu (Data Ingestion & DuckDB)

Nạp dữ liệu từ các shop vào DuckDB, kiểm tra schema, tạo catalog và áp dụng cache Semantic Views (`v_sales`, `v_orders`) và Debt macros.

In [ ]:
from time import perf_counter

t0 = perf_counter()
print(f"Bắt đầu chuẩn bị DuckDB & Catalog cho {DATASET_SPLIT}...", flush=True)
prep_report = pipeline.prepare_datasets(splits=SPLITS)
print(f"Chuẩn bị dữ liệu xong sau {perf_counter() - t0:.1f}s")

# Kiểm tra tính toàn vẹn của các DuckDB databases
validation = pipeline.validate_databases(splits=SPLITS)
print(f"Xác minh thành công {len(validation)} databases.")

# Đồng bộ báo cáo ingestion sang Drive
sync_outputs_to_drive()


## 4. Smoke Test: Chạy thử nghiệm 3 câu

In [ ]:
if DATASET_SPLIT == "train":
    print("Chạy Smoke Test 3 câu train...")
    _, _, smoke_scores, _, smoke_logs = pipeline.smoke_test()
    for row in smoke_logs:
        print(f"ID: {row.get('id')} | Type: {row.get('type')} | SQL: {row.get('sql')[:80]}...")
    sync_outputs_to_drive()
else:
    print("Đang chọn public_test: Bỏ qua smoke test, chuyển thẳng sang bước 5 để dự đoán.")


## 5. Chạy Suy Luận (Inference) / Đánh Giá (Evaluation)

In [ ]:
t_run = perf_counter()

if DATASET_SPLIT == "public_test":
    print("=== BẮT ĐẦU DỰ ĐOÁN TOÀN BỘ PUBLIC TEST ===")
    submission_path = pipeline.predict_public_test()
    print(f"\nHoàn tất dự đoán sau {perf_counter() - t_run:.1f}s!")
    print(f"File nộp bài tạm thời: {submission_path}")
else:
    print("=== BẮT ĐẦU ĐÁNH GIÁ TẬP TRAIN ===")
    _, _, train_scores, _, train_logs = pipeline.evaluate()
    print(f"\nHoàn tất đánh giá sau {perf_counter() - t_run:.1f}s!")
    print("\nThống kê kết quả train:")
    print(train_scores.status.value_counts())

# LƯU TẤT CẢ OUTPUT, LOGS, VÀ BÀI NỘP VÀO GOOGLE DRIVE
sync_outputs_to_drive()

# Tải file về máy cá nhân nếu muốn
if DATASET_SPLIT == "public_test" and IN_COLAB:
    from google.colab import files
    drive_sub = DRIVE_SAVE_DIR / "public_test" / "submission.jsonl"
    target = drive_sub if drive_sub.exists() else submission_path
    print(f"Tải file {target.name} về máy...")
    files.download(str(target))


## 6. Kiểm tra các file và Logs đã được lưu trong Google Drive

In [ ]:
import json
import pandas as pd

print(f"=== DANH SÁCH FILE VÀ LOG ĐÃ LƯU TRONG GOOGLE DRIVE ({DRIVE_SAVE_DIR}) ===\n")
drive_files = sorted(DRIVE_SAVE_DIR.rglob("*"))
for f in drive_files:
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f"✓ {f.relative_to(DRIVE_SAVE_DIR)} ({size_kb:.1f} KB)")

# Hiển thị mẫu nội dung submission nếu có
sub_file = DRIVE_SAVE_DIR / "public_test" / "submission.jsonl"
if not sub_file.exists():
    sub_file = Path(LOCAL_OUT_DIR) / "public_test" / "submission.jsonl"

if sub_file.exists():
    rows = [json.loads(line) for line in sub_file.read_text(encoding='utf-8').splitlines() if line.strip()]
    df = pd.DataFrame(rows)
    print(f"\nTổng số câu trong submission: {len(df)}")
    print("Phân bố các nhãn đáp án:")
    print(df['type'].value_counts())
    print("\n5 dòng kết quả đầu tiên:")
    display(df.head(5))

# Hiển thị mẫu log
log_file = DRIVE_SAVE_DIR / "public_test" / "logs" / "test_log.jsonl"
if not log_file.exists():
    log_file = Path(LOCAL_OUT_DIR) / "public_test" / "logs" / "test_log.jsonl"
if log_file.exists():
    sample_log = [json.loads(line) for line in log_file.read_text(encoding='utf-8').splitlines()[:1]]
    if sample_log:
        print("\nCấu trúc một bản ghi log chi tiết (lưu trong Drive):")
        print(json.dumps(sample_log[0], ensure_ascii=False, indent=2)[:500] + "...")
